In [ ]:
from sklearn.metrics import roc_auc_score

# Get predicted probabilities (needed for ROC-AUC, not just hard predictions)
y_proba_rf = rf_clf.predict_proba(X_test_tfidf)

roc_auc = roc_auc_score(
    y_test, y_proba_rf, multi_class="ovr", average="macro", labels=rf_clf.classes_
)
print("Random Forest ROC-AUC (macro, one-vs-rest):", round(roc_auc, 4))

In [ ]:
import pickle
import os

# Create the directory if it doesn't exist
os.makedirs("../models", exist_ok=True)

with open("../models/classifier.pkl", "wb") as f:
    pickle.dump(rf_clf, f)

with open("../models/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)

print("Saved classifier.pkl and tfidf_vectorizer.pkl to /models")

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_clf.fit(X_train_tfidf, y_train)

y_pred_rf = rf_clf.predict(X_test_tfidf)

print("Random Forest Accuracy:", round(accuracy_score(y_test, y_pred_rf), 4))
print("\nLogistic Regression Accuracy:", round(accuracy_score(y_test, y_pred), 4))

In [ ]:
import os

# Confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=clf.classes_)

plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=clf.classes_, yticklabels=clf.classes_)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Logistic Regression")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()

# Create the directory if it doesn't exist
output_dir = "../reports/figures"
os.makedirs(output_dir, exist_ok=True)

plt.savefig(os.path.join(output_dir, "confusion_matrix_logreg.png"), dpi=150)
plt.show()

In [ ]:
# Drop inconsistent/low-sample categories
bad_categories = ["Java Developer", "DevOps Engineer", "Testing"]
resumes_filtered = resumes[~resumes["Category"].isin(bad_categories)].reset_index(drop=True)

print("Before filtering:", resumes.shape)
print("After filtering:", resumes_filtered.shape)
print("Remaining categories:", resumes_filtered["Category"].nunique())

# Redo the split, vectorization, and training on filtered data
X = resumes_filtered["clean_text"]
y = resumes_filtered["Category"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

tfidf = TfidfVectorizer(max_features=5000, stop_words="english")
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train_tfidf, y_train)

y_pred = clf.predict(X_test_tfidf)
print("\nNew accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\n", classification_report(y_test, y_pred))

In [ ]:
print("All categories after reload:")
print(resumes["Category"].value_counts())

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

y_pred = clf.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", round(accuracy, 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Convert text to numeric vectors
tfidf = TfidfVectorizer(max_features=5000, stop_words="english")
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("TF-IDF train shape:", X_train_tfidf.shape)
print("TF-IDF test shape:", X_test_tfidf.shape)

# Train classifier
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train_tfidf, y_train)

print("\nModel trained.")

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

resumes = pd.read_csv("../data/processed/resumes_clean.csv")
resumes = resumes.dropna(subset=["clean_text", "Category"])

print("Total resumes:", resumes.shape)
print("Categories:", resumes["Category"].nunique())

X = resumes["clean_text"]
y = resumes["Category"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("\nTrain size:", X_train.shape[0])
print("Test size:", X_test.shape[0])